In [1]:
import pandas as pd
import numpy as np
import os
import re
import gc
from sklearn.decomposition import PCA
from sklearn.model_selection import KFold
from scipy import stats
from statsmodels.stats.multitest import multipletests
import time

# paths
AA_GENO = r"C:\Users\user\Downloads\GSE148375_clean\checkpoint7_snp_encoded_012.csv"
AA_META = r"C:\Users\user\Downloads\GSE148375_clean\checkpoint2_metadata_sample_filtered.csv"
MANIFEST = r"C:\Users\user\Downloads\HumanExome-12-v1-0-B.csv"
OUT_DIR = r"C:\Users\user\Downloads\GSE148375_clean"

print("Imports done.")

Imports done.


In [2]:
def strip_suffix(pid):
    return re.sub(r'_\d+$', '', pid)

# load manifest — get chromosome per probe
manifest_df = pd.read_csv(MANIFEST, skiprows=7, low_memory=False)
manifest_df["core_name"] = manifest_df["IlmnID"].map(strip_suffix)
non_autosomal = {"X", "Y", "XY", "MT", "0"}
sex_linked_cores = set(
    manifest_df.loc[manifest_df["Chr"].isin(non_autosomal), "core_name"]
)
print("Non-autosomal probes in manifest:", len(sex_linked_cores))

# load encoded SNP matrix (SNPs as rows, samples as columns)
encoded_df = pd.read_csv(AA_GENO)
probe_id_array = encoded_df["probe_id"].to_numpy()
sample_ids = encoded_df.columns[1:].tolist()
print("Total probes:", len(probe_id_array))
print("Total samples:", len(sample_ids))

# filter to autosomal probes only
keep_mask = ~np.array([strip_suffix(p) in sex_linked_cores for p in probe_id_array])
print("Autosomal probes retained:", keep_mask.sum())

# build samples x SNPs matrix (int8 to save memory)
X_snp_rows = encoded_df[encoded_df.columns[1:]].to_numpy(dtype=np.int8)[keep_mask]
X_auto = X_snp_rows.T  # now samples x SNPs
probe_ids_auto = probe_id_array[keep_mask]

del encoded_df, X_snp_rows
gc.collect()

print("X_auto shape (samples x SNPs):", X_auto.shape)
print("Memory (MB):", X_auto.nbytes / 1e6)

Non-autosomal probes in manifest: 5683
Total probes: 238927
Total samples: 3348
Autosomal probes retained: 233610
X_auto shape (samples x SNPs): (3348, 233610)
Memory (MB): 782.12628


In [3]:
# load metadata
meta_df = pd.read_csv(AA_META)
meta_df["sample_id"] = meta_df["sample_id"].astype(str)
meta_df = meta_df.set_index("sample_id").reindex(sample_ids).reset_index()

print("Metadata shape:", meta_df.shape)
print("Missing values:", meta_df[["age", "gender", "smoking_status"]].isna().sum())
print("Smoking status distribution:")
print(meta_df["smoking_status"].value_counts())

# outcome variable Y: smoking_status binary
Y = (meta_df["smoking_status"] == "Smoker").astype(np.float64).values
print("\nY shape:", Y.shape)
print("Y distribution:", np.unique(Y, return_counts=True))

# confounder variables: age + gender (binary)
age = meta_df["age"].astype(np.float64).values
gender_binary = (meta_df["gender"] == "Male").astype(np.float64).values

# standardize age (zero mean, unit variance) — important for numerical stability
age_std = (age - age.mean()) / age.std()

print("\nAge stats (standardized): mean=", age_std.mean().round(6), "std=", age_std.std().round(6))
print("Gender distribution:", np.unique(gender_binary, return_counts=True))

Metadata shape: (3348, 9)
Missing values: age               0
gender            0
smoking_status    0
dtype: int64
Smoking status distribution:
smoking_status
Non-smoker    1717
Smoker        1631
Name: count, dtype: int64

Y shape: (3348,)
Y distribution: (array([0., 1.]), array([1717, 1631]))

Age stats (standardized): mean= 0.0 std= 1.0
Gender distribution: (array([0., 1.]), array([1785, 1563]))


In [4]:
# --- EIGENSTRAT standardization (autosomal SNPs only) ---
X_float = X_auto.astype(np.float32)
del X_auto
gc.collect()

p = X_float.mean(axis=0) / 2
denom = np.sqrt(2 * p * (1 - p))
valid_mask = denom > 1e-8
print("Monomorphic SNPs excluded:", (~valid_mask).sum())
print("Informative SNPs retained:", valid_mask.sum())

X_std = (X_float[:, valid_mask] - 2 * p[valid_mask]) / denom[valid_mask]
probe_ids_valid = probe_ids_auto[valid_mask]

del X_float
gc.collect()

print("X_std shape:", X_std.shape)

# --- PCA: top 10 components (ancestry proxy) ---
pca = PCA(n_components=10, random_state=42, copy=False, svd_solver='randomized')
pcs = pca.fit_transform(X_std)

print("Explained variance ratio per PC:")
for i, v in enumerate(pca.explained_variance_ratio_):
    print(f"  PC{i+1}: {v:.4f}")

# --- Build full confounder matrix X_conf = [PC1..10, age_std, gender] ---
X_conf = np.hstack([
    pcs,                          # (3348, 10)
    age_std.reshape(-1, 1),       # (3348, 1)
    gender_binary.reshape(-1, 1)  # (3348, 1)
]).astype(np.float64)

print("\nX_conf shape:", X_conf.shape)  # expect (3348, 12)
print("Columns: PC1..PC10, age_std, gender_binary")

Monomorphic SNPs excluded: 91965
Informative SNPs retained: 141645
X_std shape: (3348, 141645)
Explained variance ratio per PC:
  PC1: 0.0056
  PC2: 0.0011
  PC3: 0.0011
  PC4: 0.0010
  PC5: 0.0010
  PC6: 0.0010
  PC7: 0.0009
  PC8: 0.0009
  PC9: 0.0009
  PC10: 0.0008

X_conf shape: (3348, 12)
Columns: PC1..PC10, age_std, gender_binary


In [5]:
def doubleml_scan(X_snps, Y, X_conf, n_folds=5, random_state=42):
    """
    Vectorized DoubleML PLR scan across all SNPs.
    Returns theta and p-value per SNP.
    """
    n, n_snps = X_snps.shape
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=random_state)

    D_resid = np.zeros_like(X_snps)
    Y_resid = np.zeros(n)

    Xc = np.column_stack([np.ones(n), X_conf])  # add intercept

    for train_idx, test_idx in kf.split(Xc):
        Xc_tr, Xc_te = Xc[train_idx], Xc[test_idx]

        # residualize Y
        coef_Y = np.linalg.lstsq(Xc_tr, Y[train_idx], rcond=None)[0]
        Y_resid[test_idx] = Y[test_idx] - Xc_te @ coef_Y

        # residualize all SNPs at once (vectorized)
        coef_D = np.linalg.lstsq(Xc_tr, X_snps[train_idx], rcond=None)[0]
        D_resid[test_idx] = X_snps[test_idx] - Xc_te @ coef_D

    # center residuals
    Yr = Y_resid - Y_resid.mean()
    Dr = D_resid - D_resid.mean(axis=0)
    del D_resid

    # compute theta and p-value per SNP (algebraic shortcut, no large intermediate matrix)
    sum_DY = (Dr * Yr[:, None]).sum(axis=0)
    sum_DD = (Dr ** 2).sum(axis=0)
    sum_YY = (Yr ** 2).sum()

    theta = sum_DY / sum_DD
    ssr = sum_YY - (sum_DY ** 2) / sum_DD
    sigma2 = ssr / (n - 2)
    se = np.sqrt(sigma2 / sum_DD)
    t_stat = theta / se
    pvals = 2 * (1 - stats.t.cdf(np.abs(t_stat), df=n - 2))

    return theta, pvals

# single test iteration
print("Running single DoubleML iteration (test run)...")
start = time.time()
theta_test, pvals_test = doubleml_scan(X_std, Y, X_conf, random_state=0)
elapsed = time.time() - start
print(f"Completed in {elapsed:.1f}s")

# apply BH-FDR at q<0.10
reject, pvals_corr, _, _ = multipletests(pvals_test, alpha=0.10, method='fdr_bh')
print(f"\nSNPs passing BH-FDR q<0.10: {reject.sum()}")
print(f"SNPs passing BH-FDR q<0.05: {(pvals_corr < 0.05).sum()}")
print(f"SNPs passing BH-FDR q<0.20: {(pvals_corr < 0.20).sum()}")
print(f"Min corrected p-value: {pvals_corr.min():.6f}")

Running single DoubleML iteration (test run)...
Completed in 83.0s

SNPs passing BH-FDR q<0.10: 0
SNPs passing BH-FDR q<0.05: 0
SNPs passing BH-FDR q<0.20: 0
Min corrected p-value: 0.471505


In [6]:
# confirm: how many SNPs pass raw p<0.001 in this test iteration
# (with augmented confounders including age + gender)
n_pass_001 = (pvals_test < 0.001).sum()
n_pass_0001 = (pvals_test < 0.0001).sum()
n_pass_01 = (pvals_test < 0.01).sum()

print(f"Raw p < 0.01:   {n_pass_01} SNPs")
print(f"Raw p < 0.001:  {n_pass_001} SNPs")
print(f"Raw p < 0.0001: {n_pass_0001} SNPs")

Raw p < 0.01:   1257 SNPs
Raw p < 0.001:  117 SNPs
Raw p < 0.0001: 13 SNPs


In [7]:
n_repeats = 30
threshold = 0.001
n_snps = X_std.shape[1]

significant_counts = np.zeros(n_snps, dtype=np.int32)

print(f"Running {n_repeats} stability selection repeats (p<{threshold})...")
start = time.time()

for rep in range(n_repeats):
    _, pvals_rep = doubleml_scan(X_std, Y, X_conf, random_state=rep)
    significant_counts += (pvals_rep < threshold).astype(np.int32)
    if (rep + 1) % 5 == 0:
        elapsed = time.time() - start
        print(f"  Repeat {rep+1}/{n_repeats} — elapsed {elapsed/60:.1f} min")

total_elapsed = time.time() - start
print(f"\nTotal time: {total_elapsed/60:.1f} minutes")

stability_fraction = significant_counts / n_repeats

# summary
print("\nStability fraction distribution:")
print(pd.Series(stability_fraction).describe())
for thresh in [0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:
    n = (stability_fraction >= thresh).sum()
    print(f"  >= {thresh:.0%} stability: {n} SNPs")

# save
stability_df = pd.DataFrame({
    "probe_id": probe_ids_valid,
    "stability_fraction": stability_fraction,
    "n_significant_repeats": significant_counts
}).sort_values("stability_fraction", ascending=False)

stability_df.to_csv(os.path.join(OUT_DIR, "v2_stability_results.csv"), index=False)
print("\nSaved stability results.")

Running 30 stability selection repeats (p<0.001)...
  Repeat 5/30 — elapsed 6.3 min
  Repeat 10/30 — elapsed 12.5 min
  Repeat 15/30 — elapsed 18.5 min
  Repeat 20/30 — elapsed 24.3 min
  Repeat 25/30 — elapsed 30.1 min
  Repeat 30/30 — elapsed 35.8 min

Total time: 35.8 minutes

Stability fraction distribution:
count    141645.000000
mean          0.000871
std           0.027271
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max           1.000000
dtype: float64
  >= 50% stability: 116 SNPs
  >= 60% stability: 111 SNPs
  >= 70% stability: 108 SNPs
  >= 80% stability: 100 SNPs
  >= 90% stability: 92 SNPs
  >= 100% stability: 67 SNPs

Saved stability results.


In [1]:
import pandas as pd
import numpy as np
import os
import re

EA_OUT = r"C:\Users\user\Downloads\GSE148812_clean"
MANIFEST = r"C:\Users\user\Downloads\HumanExome-12-v1-0-B.csv"

manifest_df = pd.read_csv(MANIFEST, skiprows=7, low_memory=False)
manifest_df["core_name"] = manifest_df["IlmnID"].map(lambda x: re.sub(r'_\d+$', '', x))
non_autosomal = {"X", "Y", "XY", "MT", "0"}
chr_lookup = manifest_df.set_index("core_name")["Chr"]
pos_lookup_m = manifest_df.set_index("core_name")["MapInfo"]

stability_df_ea = pd.read_csv(os.path.join(EA_OUT, "v2_stability_results.csv"))

shortlist_ea_v2 = stability_df_ea[stability_df_ea["stability_fraction"] == 1.0].copy()
shortlist_ea_v2["core_name"] = shortlist_ea_v2["probe_id"].map(lambda x: re.sub(r'_\d+$', '', x))

pos_lookup = manifest_df.set_index("core_name")[["Chr", "MapInfo"]]
shortlist_ea_v2 = shortlist_ea_v2.merge(pos_lookup, on="core_name", how="left")
shortlist_ea_v2 = shortlist_ea_v2[~shortlist_ea_v2["Chr"].isin(non_autosomal)].copy()
print("100% stable SNPs:", len(shortlist_ea_v2))

# load genotype vectors for LD pruning
EA_GENO = r"C:\Users\user\Downloads\GSE148812_clean\checkpoint7_snp_encoded_012.csv"
encoded_df_ea = pd.read_csv(EA_GENO)
probe_rows = encoded_df_ea[encoded_df_ea["probe_id"].isin(set(shortlist_ea_v2["probe_id"]))].copy()
probe_rows = probe_rows.set_index("probe_id").reindex(shortlist_ea_v2["probe_id"].tolist())
X_shortlist_ea = probe_rows.to_numpy(dtype=np.float64).T
probe_id_to_idx_ea = {pid: i for i, pid in enumerate(shortlist_ea_v2["probe_id"].tolist())}
del encoded_df_ea, probe_rows

def get_geno_ea(pid):
    return X_shortlist_ea[:, probe_id_to_idx_ea[pid]]

def greedy_ld_prune(df, r2_thresh=0.2, window_bp=1_000_000):
    sorted_df = df.sort_values("stability_fraction", ascending=False).reset_index(drop=True)
    retained = []
    removed = set()
    for i, row_i in sorted_df.iterrows():
        pid_i = row_i["probe_id"]
        if pid_i in removed:
            continue
        retained.append(pid_i)
        g_i = get_geno_ea(pid_i)
        for j, row_j in sorted_df.iloc[i+1:].iterrows():
            pid_j = row_j["probe_id"]
            if pid_j in removed:
                continue
            if row_j["Chr"] != row_i["Chr"]:
                continue
            if abs(row_j["MapInfo"] - row_i["MapInfo"]) > window_bp:
                continue
            r2 = np.corrcoef(g_i, get_geno_ea(pid_j))[0, 1] ** 2
            if r2 > r2_thresh:
                removed.add(pid_j)
    return retained

retained_ea_v2 = greedy_ld_prune(shortlist_ea_v2, r2_thresh=0.2)
shortlist_pruned_ea_v2 = shortlist_ea_v2[
    shortlist_ea_v2["probe_id"].isin(retained_ea_v2)
].copy().sort_values(["Chr", "MapInfo"]).reset_index(drop=True)

print(f"After LD pruning: {len(shortlist_pruned_ea_v2)} SNPs")
shortlist_pruned_ea_v2.to_csv(os.path.join(EA_OUT, "v2_shortlist_ld_pruned_v2.csv"), index=False)
print(shortlist_pruned_ea_v2[["probe_id", "Chr", "MapInfo"]].to_string())

100% stable SNPs: 57
After LD pruning: 52 SNPs
                            probe_id Chr      MapInfo
0           exm6967-0_B_F_1919124627   1    3417762.0
1          exm71047-0_B_R_1921357564   1   85020695.0
2         exm144193-0_B_R_2060145058   1  207195568.0
3         exm834716-0_B_R_1920965043  10   75523634.0
4         exm850504-0_B_R_1921041917  10  102824292.0
5   exm-rs2981575-131_T_R_1990491203  10  123346116.0
6         exm906375-0_B_R_1918200120  11   48238998.0
7         exm935491-0_T_R_1918372056  11   68562288.0
8   exm-rs1678542-131_B_R_1990479916  12   57968715.0
9        exm1047664-0_B_F_1922520599  12  123661295.0
10       exm1071911-0_B_F_2060125197  13   73335638.0
11       exm1094587-0_B_R_1922736292  14   25043951.0
12       exm1106910-0_T_F_1922747987  14   64498037.0
13       exm2252100-0_T_F_1975263808  15   31376365.0
14       exm1168225-0_T_R_1922834178  15   63937724.0
15       exm1247860-0_T_R_1921678048  16   67210855.0
16       exm1381429-0_B_F_192162893

In [10]:
import json
from causallearn.utils.PCUtils.BackgroundKnowledge import BackgroundKnowledge

# build PC input matrix: 56 SNPs + smoking_status
pruned_ids = shortlist_pruned["probe_id"].tolist()
col_names_pc = pruned_ids + ["smoking_status"]

encoded_df_56 = pd.read_csv(AA_GENO)
probe_rows_56 = encoded_df_56[encoded_df_56["probe_id"].isin(set(pruned_ids))].copy()
probe_rows_56 = probe_rows_56.set_index("probe_id").reindex(pruned_ids)
X_pc = probe_rows_56.to_numpy(dtype=np.float64).T  # samples x 56

X_pc_full = np.hstack([X_pc, Y.reshape(-1, 1)])
print("PC input shape:", X_pc_full.shape)  # expect (3348, 57)

# save for fresh notebook
np.save(os.path.join(OUT_DIR, "v2_pc_input.npy"), X_pc_full)
with open(os.path.join(OUT_DIR, "v2_pc_col_names.json"), "w") as f:
    json.dump(col_names_pc, f)

print("Saved PC input matrix and column names.")
print("smoking_status index:", col_names_pc.index("smoking_status"))

PC input shape: (3348, 57)
Saved PC input matrix and column names.
smoking_status index: 56
